# Week 2: Python foundations II (comprehensions, files, modules, tests, the debugger)

**Budget:** about 10 hours.

| Session | Topic | Time |
|---|---|---|
| Day 1 | Tuples, sets, comprehensions, sorting | 2.5 h |
| Day 2 | Files, CSV data, exceptions | 2.5 h |
| Day 3 | From notebook to module: your first `pytest` test suite | 2.5 h |
| Day 4 | The VS Code debugger, plus Git branches and `.gitignore` | 2 h |
| Review | Spaced recall (weeks 1 and 2), blank-page rebuild, commit | 0.5 h |

### Done when
- [ ] Every test cell prints ✅.
- [ ] `../work/week02/` contains `text_utils.py` and `test_text_utils.py` with **at least 6 tests that you wrote yourself**, and `pytest` passes.
- [ ] You have used the VS Code debugger to step through a function and can describe what you saw.
- [ ] You have worked on a branch `week02`, merged it into `main`, and pushed.

---
# Day 1: Tuples, sets, comprehensions, sorting (2.5 h)

**Tuples** are like lists but can't be changed. They're used for fixed groups of values, such as `(word, count)`.
**Sets** hold unique items and answer "is x in here?" instantly. They're ideal for vocabularies and for checking which documents contain a word.

In [ ]:
point = (3, 4)
x, y = point                       # "unpacking"
print(x, y)

a = {"mri", "ct", "xray"}
b = {"ct", "pet"}
print(a & b)    # intersection: in both
print(a | b)    # union: in either
print(a - b)    # difference: in a but not in b
print("mri" in a)
print(set(["a", "b", "a"]))        # duplicates disappear

### Comprehensions
A **comprehension** builds a list, set, or dict in one line from a loop and an optional condition. Read it as: "*give me* `expression` *for each* `x` *in* `items` *if* `condition`".

In [ ]:
nums = [1, 2, 3, 4, 5, 6]
squares = [n * n for n in nums]
evens = [n for n in nums if n % 2 == 0]
lengths = {w: len(w) for w in ["mri", "scan", "x"]}
first_letters = {w[0] for w in ["mri", "mass", "ct"]}
print(squares, evens, lengths, first_letters)

# the same as `evens` above, written as an ordinary loop:
evens_loop = []
for n in nums:
    if n % 2 == 0:
        evens_loop.append(n)
print(evens_loop == evens)

### Sorting with `key`
`sorted(items, key=f)` sorts items by `f(item)`. The original list is not changed.

In [ ]:
words = ["pneumonia", "ct", "lung", "x"]
print(sorted(words))                 # alphabetical
print(sorted(words, key=len))        # by length
print(sorted(words, key=lambda w: (-len(w), w)))   # longest first, then alphabetical

### Exercise 2.1: `squares_of_evens`
Using **one list comprehension**, return the squares of the even numbers, in their original order.

In [ ]:
def squares_of_evens(nums):
    raise NotImplementedError

In [ ]:
assert squares_of_evens([1, 2, 3, 4]) == [4, 16]
assert squares_of_evens([]) == []
print("✅ 2.1 passed")

### Exercise 2.2: `unique_sorted`
Return the unique tokens, sorted alphabetically, as a list.

In [ ]:
def unique_sorted(tokens):
    raise NotImplementedError

In [ ]:
assert unique_sorted(["b", "a", "b"]) == ["a", "b"]
assert unique_sorted([]) == []
print("✅ 2.2 passed")

### Exercise 2.3: `jaccard`, your first similarity measure
The **Jaccard similarity** of two token lists is $\dfrac{|A \cap B|}{|A \cup B|}$, where $A$ and $B$ are their *sets* of tokens. It's 1 for identical vocabularies and 0 when they share nothing. Return `0.0` if both lists are empty. In Month 3 you'll see why search engines use TF-IDF and BM25 instead, but Jaccard is the starting point.

In [ ]:
def jaccard(tokens_a, tokens_b):
    raise NotImplementedError

In [ ]:
assert jaccard(["a", "b"], ["b", "c"]) == 1 / 3
assert jaccard(["a"], ["a", "a"]) == 1.0
assert jaccard([], []) == 0.0
assert jaccard(["x"], []) == 0.0
print("✅ 2.3 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Convert both lists with `set(...)`.

</details>

<details><summary>Hint 2</summary>

Use `&` and `|`, and check for an empty union *before* dividing.

</details>


### Exercise 2.4: `invert`
Given a dict `word -> id`, return a dict `id -> word` using a **dict comprehension**. You'll need this for vocabularies: `encode` maps words to ids and `decode` maps them back.

In [ ]:
def invert(d):
    raise NotImplementedError

In [ ]:
assert invert({"a": 0, "b": 1}) == {0: "a", 1: "b"}
print("✅ 2.4 passed")

---
# Day 2: Files, CSV data, exceptions (2.5 h)

Real data lives in files, and real data is messy. We'll use **`pathlib`** for paths and the **`csv`** module to read rows. The cell below creates a small, *synthetic* study file. (It's invented; you will never need real patient data in this course.)

In [ ]:
from pathlib import Path

DATA = Path("../work/week02/data")
DATA.mkdir(parents=True, exist_ok=True)
csv_path = DATA / "scores.csv"
csv_path.write_text(
    "participant,group,score\n"
    "p01,control,7.5\n"
    "p02,treatment,8.0\n"
    "p03,control,\n"
    "p04,treatment,9.5\n"
    "p05,control,6.5\n"
    "p06,treatment,n/a\n"
)
print(csv_path.read_text())

In [ ]:
import csv

with open(csv_path) as f:            # 'with' closes the file automatically
    reader = csv.DictReader(f)        # each row becomes a dict keyed by the header
    for row in reader:
        print(row)

### Exceptions: failing loudly and handling failures on purpose
- `raise ValueError("message")` stops the program when the input makes no sense. That's much better than quietly returning a wrong number.
- `try: ... except ValueError: ...` handles a failure you *expect* and know how to deal with.

Only catch the exceptions you expect. A bare `except:` hides bugs.

In [ ]:
def parse_age(s):
    age = int(s)                 # raises ValueError for "abc"
    if age < 0:
        raise ValueError(f"age cannot be negative: {age}")
    return age

for s in ["42", "abc", "-1"]:
    try:
        print(s, "->", parse_age(s))
    except ValueError as e:
        print(s, "-> rejected:", e)

### Exercise 2.5: `safe_float`
Return `float(s)`, or `None` if `s` can't be converted. Use `try`/`except`.

In [ ]:
def safe_float(s):
    raise NotImplementedError

In [ ]:
assert safe_float("7.5") == 7.5
assert safe_float("") is None
assert safe_float("n/a") is None
print("✅ 2.5 passed")

### Exercise 2.6: `load_scores`
Read the CSV at `path` and return a tuple `(rows, n_skipped)`:
- `rows` is a list of dicts `{"participant": str, "group": str, "score": float}`, one for each row with a valid score,
- `n_skipped` is how many rows were skipped because the score was missing or invalid.

**Always count and report what you drop.** Dropping data without saying so is a classic source of bias and of results nobody can reproduce.

In [ ]:
import csv

def load_scores(path):
    raise NotImplementedError

In [ ]:
rows, skipped = load_scores(csv_path)
assert skipped == 2
assert len(rows) == 4
assert rows[0] == {"participant": "p01", "group": "control", "score": 7.5}
print("✅ 2.6 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Open the file with `with open(path) as f:` and use `csv.DictReader(f)`.

</details>

<details><summary>Hint 2</summary>

For each row, `value = safe_float(row["score"])`. If it's `None`, add one to the skip counter and `continue`.

</details>


### Exercise 2.7: `mean_by_group`
Return a dict `group -> mean score`.

In [ ]:
def mean_by_group(rows):
    raise NotImplementedError

In [ ]:
m = mean_by_group(rows)
assert m == {"control": 7.0, "treatment": 8.75}
print("✅ 2.7 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Build a dict `group -> list of scores` first, then a second dict of means. `setdefault` or `.get(g, [])` will help.

</details>


✍️ *The treatment group has a higher mean. Can you conclude that the treatment works? Give two reasons why not yet. (We'll come back to this in Week 8 with statistical tests.)* (double-click to edit)



---
# Day 3: From notebook to module, and your first test suite (2.5 h)

Notebooks are great for *exploring*. Code you want to **reuse** belongs in `.py` **modules**, and code you want to **trust** needs **tests**. A test is a small, executable claim such as "`jaccard` of identical lists is 1.0". `pytest` finds every function whose name starts with `test_` and runs it.

Here's a complete example. The cells below write two files and then run pytest on them.

In [ ]:
%%writefile ../work/week02/stats_utils.py
def mean(values):
    """Return the arithmetic mean. Raises ValueError on empty input."""
    if not values:
        raise ValueError("mean of empty list")
    return sum(values) / len(values)

In [ ]:
%%writefile ../work/week02/test_stats_utils.py
import pytest
from stats_utils import mean


def test_mean_simple():
    assert mean([1, 2, 3]) == 2


def test_mean_single_value():
    assert mean([5]) == 5


def test_mean_empty_raises():
    with pytest.raises(ValueError):
        mean([])

In [ ]:
import subprocess, sys

def run_pytest(folder):
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", folder], capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-1000:])
    return r.returncode

assert run_pytest("../work/week02/test_stats_utils.py") == 0

### Exercise 2.8: Build `text_utils.py` and its tests, in VS Code
This time, **don't** use `%%writefile`. Create the files in VS Code, the way you'd work on a real project.

1. Create `work/week02/text_utils.py` containing `tokenize`, `word_counts`, `most_common` (from Week 1) and `jaccard` (from today). Give each one a docstring. Copying your *own* earlier code is fine.
2. Create `work/week02/test_text_utils.py` with **at least 6 tests**. Cover normal cases **and edge cases**: an empty string, text that's only punctuation, ties in `most_common`, identical documents in `jaccard`.
3. Run the checker cell below. It runs your tests, and then a few of mine.

Good tests are **specific** (each one checks one thing) and **named after what they check**, e.g. `test_tokenize_removes_punctuation`.

In [ ]:
# Nothing to write here. Create the two files in VS Code, then run the next cell.

In [ ]:
import importlib, sys
sys.path.insert(0, "../work/week02")
import text_utils
importlib.reload(text_utils)

assert run_pytest("../work/week02/test_text_utils.py") == 0, "your own tests fail"
n_tests = open("../work/week02/test_text_utils.py").read().count("def test_")
assert n_tests >= 6, f"only {n_tests} tests; write at least 6"
# my hidden checks:
assert text_utils.tokenize("Hi, hi!") == ["hi", "hi"]
assert text_utils.most_common({"b": 1, "a": 1}, 1) == [("a", 1)]
assert text_utils.jaccard([], []) == 0.0
print(f"✅ 2.8 passed with {n_tests} of your own tests")

---
# Day 4: The debugger, Git branches, and `.gitignore` (2 h)

### 4.1 Using the VS Code debugger
`print` debugging works, but a **debugger** lets you pause the program and look at *every* variable. In VS Code:
1. Click in the margin to the left of a line number to set a **breakpoint** (a red dot).
2. Click the arrow next to the cell's Run button and choose **Debug Cell**.
3. Execution stops at your breakpoint. Look at the **Variables** panel.
4. **Step Over** (F10) runs one line. **Step Into** (F11) goes inside a function call. **Continue** (F5) runs to the next breakpoint.

Use the debugger, not guesswork, to fix the two functions below. Put a breakpoint inside each one, debug the test cell, and watch the variables change.

#### Debug 1: `median` (two bugs)

In [ ]:
def median(values):
    n = len(values)
    mid = n // 2
    if n % 2 == 1:
        return values[mid]
    return (values[mid] + values[mid + 1]) / 2

In [ ]:
assert median([3, 1, 2]) == 2
assert median([4, 1, 3, 2]) == 2.5
assert median([1, 2, 3, 4]) == 2.5
print("✅ median fixed")

✍️ *What I saw in the debugger, and the two bugs: ...* (double-click to edit)



#### Debug 2: `merge_counts` (it changes its input, which it shouldn't)

In [ ]:
def merge_counts(a, b):
    """Return a NEW dict with the counts from a and b added together."""
    result = a
    for k, v in b.items():
        result[k] = result.get(k, 0) + v
    return result

In [ ]:
x = {"a": 1}
y = {"a": 2, "b": 1}
z = merge_counts(x, y)
assert z == {"a": 3, "b": 1}
assert x == {"a": 1}, f"input was modified: {x}"
print("✅ merge_counts fixed")

✍️ *The bug is related to Week 1, exercise 1.4, because ...* (double-click to edit)



### 4.2 Git: `.gitignore` and branches
Some files should **never** be committed: virtual environments, caches, large datasets, and secrets. Create a file called `.gitignore` in the root of your repo containing:
```
.venv/
__pycache__/
.ipynb_checkpoints/
.pytest_cache/
data/
*.pyc
```
**Branches** let you work on something without touching `main`. Professionals never work directly on `main`. From now on, every week works like this:
```bash
git switch -c week02          # create a new branch and move to it
# ... work, then commit often with clear messages ...
git add .
git commit -m "Add text_utils module with tests"
git push -u origin week02
```
Then, on GitHub, open a **Pull Request** from `week02` to `main`, read your own diff, and click **Merge**. Back in the terminal:
```bash
git switch main
git pull
```
Good commit messages are in the imperative and describe *what changed*: "Add jaccard similarity with tests", not "stuff" or "fixed".

---
# Review (30 min)

### Blank-page rebuild
Without looking, write `jaccard` and **two tests** for it in the cell below, as plain functions called at the end.

In [ ]:
def jaccard2(a, b):
    raise NotImplementedError

def test_jaccard2_identical():
    raise NotImplementedError

def test_jaccard2_disjoint():
    raise NotImplementedError

test_jaccard2_identical(); test_jaccard2_disjoint()
assert jaccard2(["a", "b"], ["b"]) == 0.5
print("✅ rebuild passed")

### Spaced recall (no peeking)
**From Week 1**
1. Why does `def f(x, items=[])` cause bugs?
2. What does `sorted(d.items(), key=lambda kv: (-kv[1], kv[0]))` do?

**From this week**

3. What's the difference between a list, a tuple, and a set? Give one AI-related use for each.
4. Why should `load_scores` report how many rows it skipped?
5. When should you catch an exception, and when should you let it crash?
6. What makes a good test name?

✍️ *Your answer:* (double-click to edit)



### Commit, log, send
Commit on your `week02` branch, open a PR, merge it, and update `LOG.md` (hours, hints you used, what was hard).
**Send me** the link to your merged PR. I'll review your tests in particular: whether they really check the edge cases.